# Workspace RayCluster + job client — Ray Data

Topic 2 uses the CodeFlare **workspace cluster** pattern that works reliably from an OpenShift AI workbench:

1. Authenticate to the OpenShift API
2. Create a `RayCluster` with `Cluster` + `ClusterConfiguration`
3. Submit work with `cluster.job_client` (Ray Jobs API on the head)
4. Tear the cluster down when finished

Runs `scale_data.py` on KubeRay workers.

Official docs: [Running Ray workloads from Jupyter](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.4/html/working_with_distributed_workloads/running-ray-based-distributed-workloads_distributed-workloads)

In [ ]:
# JupyterLab cwd is usually extras/notebooks — find the repo for working_dir.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Get **server** and **token** from the OpenShift Console (not from `oc whoami` inside the workbench):

1. OpenShift Console → your username → **Copy login command** → Display token
2. Paste values below

Lab clusters with self-signed API certs need `verify_ssl=False`.

In [ ]:
from kube_authkit import AuthConfig, get_k8s_client
from codeflare_sdk import set_api_client, list_local_queues

# Console → your username → Copy login command → Display token
OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

auth_config = AuthConfig(
    method="openshift",
    k8s_api_host=OPENSHIFT_SERVER.strip(),
    token=OPENSHIFT_TOKEN.strip(),
    verify_ssl=False,  # lab/self-signed API certs
)
set_api_client(get_k8s_client(config=auth_config))

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)

## Create the RayCluster

CPU-only config (no GPU fields). Kueue admits via `local_queue`.

In [ ]:
from codeflare_sdk import Cluster, ClusterConfiguration

cluster = Cluster(
    ClusterConfiguration(
        name="ray-workshop-data",
        namespace=NAMESPACE,
        local_queue=LOCAL_QUEUE,
        num_workers=2,
        head_cpu_requests=1,
        head_cpu_limits=2,
        head_memory_requests=2,
        head_memory_limits=4,
        worker_cpu_requests=1,
        worker_cpu_limits=2,
        worker_memory_requests=2,
        worker_memory_limits=4,
        write_to_file=False,
    )
)

cluster.apply()
cluster.wait_ready()
cluster.details()

## Submit with `job_client`

Packages the repo as `working_dir` and runs `scale_data.py` on the cluster.

In [ ]:
import time

client = cluster.job_client

submission_id = client.submit_job(
    entrypoint="python extras/scripts/scale_data.py",
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "pip": ["pyarrow>=14", "pandas>=2"],
        "env_vars": {
            "INPUT_PATH": "extras/data/iris.csv",
            "OUTPUT_DIR": "/tmp/ray-workshop-output/iris",
        },
    },
)
print(f"Submitted: {submission_id}")

terminal = {"SUCCEEDED", "FAILED", "STOPPED"}
deadline = time.time() + 900

while time.time() < deadline:
    status = client.get_job_status(submission_id)
    print(f"Job {submission_id}: {status}")
    if str(status).split(".")[-1].upper() in terminal:
        break
    time.sleep(15)
else:
    raise TimeoutError(f"Timed out waiting for job {submission_id}")

print(client.get_job_logs(submission_id))

## Tear down

Release the RayCluster when you are done (or leave it up for Topic 3).

In [ ]:
cluster.down()
print("Cluster down.")

## Verify

1. In the notebook output, look for `Done. Wrote N parquet file(s)`.
2. Optional CLI while the cluster was up:

```sh
oc get raycluster,pods -n ray-workshop
oc logs -n ray-workshop -l ray.io/cluster=ray-workshop-data,ray.io/node-type=head -c ray-head --tail=80
```

Jobs submitted via `job_client` appear in the **Ray Dashboard → Jobs** tab, not always as Kubernetes `RayJob` CRs.